# Inside the Chinook Deep Agent

A hands-on tutorial for learning how this text-to-SQL agent works from the inside out.

By the end, you will be able to:

1. explain the coordinator/subagent architecture;
2. inspect the OSI semantic model, prompts, memory, and skills;
3. test the SQL safety boundary and result artifact store;
4. construct the real Deep Agent used by the application;
5. observe a LangGraph human-in-the-loop interrupt;
6. approve, edit, or reject generated SQL; and
7. relate the notebook flow to the FastAPI and Streamlit application.

> **Cost and safety:** Building the agent does not call OpenAI. The live exercise is off by default and is clearly marked. SQL is still parsed, opened read-only, authorizer-protected, timed out, capped at 500 rows, and paused for review.

## 1. Start with the mental model

```text
User question
     │
     ▼
Coordinator Deep Agent
  ├─ conversation memory: AGENTS.md
  ├─ safe tool: get_saved_result
  └─ task → isolated text-to-SQL subagent
                 ├─ semantic/chinook.osi.yaml (primary context)
                 ├─ query-writing skill
                 ├─ schema-exploration skill
                 ├─ read-only schema/checker fallback tools
                 └─ execute_sql
                         │
                         ▼
                   HITL interrupt
                  approve/edit/reject
                         │
                         ▼
               guarded SQLite execution
                 ├─ ≤10 rows to model
                 └─ ≤500 rows to ResultStore/UI
```

The important design idea is **separation of responsibilities**. The coordinator manages the conversation. The specialist grounds itself and writes SQL. A deterministic executor enforces safety. The human owns the final execution decision. Large data lives outside model context.

## 2. Notebook setup

Run this notebook from the `data-analyst-agent` directory. The setup cell also handles being launched from the repository root. It loads `.env` but never prints your API key.

In [1]:
from pathlib import Path
import importlib.metadata as metadata
import inspect
import json
import os
import sys
import uuid

import pandas as pd
import yaml
from IPython.display import Markdown, display
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    candidate = PROJECT_ROOT / "data-analyst-agent"
    if (candidate / "pyproject.toml").is_file():
        PROJECT_ROOT = candidate

assert (PROJECT_ROOT / "text2sql_agent").is_dir(), (
    "Launch Jupyter from data-analyst-agent or its parent repository."
)
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project: {PROJECT_ROOT}")
print(f"OPENAI_API_KEY configured: {bool(os.getenv('OPENAI_API_KEY'))}")

Project: /Users/charlie/Repos/deepagents-agents/data-analyst-agent
OPENAI_API_KEY configured: True


In [2]:
from text2sql_agent.config import Settings

settings = Settings()
packages = [
    "deepagents", "langchain", "langgraph", "langchain-openai",
    "sqlglot", "fastapi", "streamlit",
]
versions = {
    package: metadata.version(package)
    for package in packages
    if package in {dist.metadata["Name"] for dist in metadata.distributions()}
}
display(pd.DataFrame(versions.items(), columns=["package", "version"]))
print(f"Model: {settings.model}")
print(f"Database: {settings.database_path}")
print(f"Semantic model: {settings.semantic_model_path}")
print(f"Readiness issues: {settings.readiness_errors() or 'none'}")

,package,version
0,deepagents,0.6.12
1,langchain,1.3.12
2,langgraph,1.2.6
3,langchain-openai,1.3.5
4,sqlglot,30.12.0
5,fastapi,0.139.2
6,streamlit,1.59.2


Model: gpt-5.4-mini
Database: /Users/charlie/Repos/deepagents-agents/data-analyst-agent/chinook.db
Semantic model: /Users/charlie/Repos/deepagents-agents/data-analyst-agent/semantic/chinook.osi.yaml
Readiness issues: none


## 3. Semantic grounding: why the agent reads OSI first

Without a semantic layer, an LLM may repeatedly probe table schemas, guess join paths, or confuse invoice totals with line-level revenue. The OSI file gives it a compact contract containing physical sources, friendly concepts, keys, join relationships, synonyms, metrics, and AI instructions.

The OSI file is **primary context**. Live schema tools are fallback evidence when the semantic file is ambiguous or appears inconsistent.

In [3]:
with settings.semantic_model_path.open() as handle:
    osi_document = yaml.safe_load(handle)

model = osi_document["semantic_model"][0]
print("OSI version:", osi_document["version"])
print("Semantic model:", model["name"])
print("Datasets:", len(model["datasets"]))
print("Relationships:", len(model["relationships"]))
print("Canonical metrics:", len(model["metrics"]))

OSI version: 0.1.1
Semantic model: chinook_music_store
Datasets: 11
Relationships: 11
Canonical metrics: 6


In [4]:
dataset_rows = [
    {
        "semantic_name": dataset["name"],
        "physical_table": dataset["source"],
        "primary_key": ", ".join(dataset["primary_key"]),
        "field_count": len(dataset["fields"]),
    }
    for dataset in model["datasets"]
]
display(pd.DataFrame(dataset_rows))

relationship_rows = [
    {
        "relationship": relation["name"],
        "from": f"{relation['from']}.{','.join(relation['from_columns'])}",
        "to": f"{relation['to']}.{','.join(relation['to_columns'])}",
    }
    for relation in model["relationships"]
]
display(pd.DataFrame(relationship_rows))

,semantic_name,physical_table,primary_key,field_count
0,artists,Artist,artist_id,2
1,albums,Album,album_id,3
2,employees,Employee,employee_id,15
3,customers,Customer,customer_id,13
4,genres,Genre,genre_id,2
5,invoices,Invoice,invoice_id,9
6,media_types,MediaType,media_type_id,2
7,tracks,Track,track_id,9
8,invoice_lines,InvoiceLine,invoice_line_id,5
9,playlists,Playlist,playlist_id,2


,relationship,from,to
0,albums_to_artists,albums.artist_id,artists.artist_id
1,employees_to_managers,employees.reports_to,employees.employee_id
2,customers_to_support_reps,customers.support_rep_id,employees.employee_id
3,invoices_to_customers,invoices.customer_id,customers.customer_id
4,tracks_to_albums,tracks.album_id,albums.album_id
5,tracks_to_media_types,tracks.media_type_id,media_types.media_type_id
6,tracks_to_genres,tracks.genre_id,genres.genre_id
7,invoice_lines_to_invoices,invoice_lines.invoice_id,invoices.invoice_id
8,invoice_lines_to_tracks,invoice_lines.track_id,tracks.track_id
9,playlist_tracks_to_playlists,playlist_tracks.playlist_id,playlists.playlist_id


In [5]:
metric_rows = []
for metric in model["metrics"]:
    dialect = metric["expression"]["dialects"][0]
    metric_rows.append({
        "metric": metric["name"],
        "expression": dialect["expression"],
        "meaning": metric["description"],
    })
display(pd.DataFrame(metric_rows))

print("Global AI instructions:\n")
print(model["ai_context"]["instructions"])

,metric,expression,meaning
0,total_revenue,SUM(invoices.Total),Canonical revenue summed once per invoice.
1,line_revenue,SUM(invoice_lines.UnitPrice * invoice_lines.Qu...,Revenue calculated from individual invoice lines.
2,units_sold,SUM(invoice_lines.Quantity),Total track units purchased.
3,invoice_count,COUNT(DISTINCT invoices.InvoiceId),Number of distinct invoices.
4,customer_count,COUNT(DISTINCT customers.CustomerId),Number of distinct customers.
5,track_count,COUNT(DISTINCT tracks.TrackId),Number of distinct catalog tracks.


Global AI instructions:

Read this model before writing SQL. Use the exact physical table and column names in each source/expression. Invoice.Total is the canonical invoice revenue; InvoiceLine.UnitPrice * InvoiceLine.Quantity is the canonical line-level revenue. Join only through declared relationships. SQLite dates are stored as timestamp-like text. Unless the user asks for more rows, return five rows. Never infer sales from playlist membership.



## 4. The context stack: prompt, memory, and skills

These three mechanisms solve different problems:

| Mechanism | What it contains | When it is available |
|---|---|---|
| System prompt | Role, delegation rules, output behavior | Always |
| `AGENTS.md` memory | Project-wide operating policy | Loaded into coordinator context |
| Skills | Focused procedures and domain heuristics | Progressively loaded when relevant |

Custom Deep Agent subagents do **not** automatically inherit coordinator skills. This project explicitly assigns both SQL skills to the `text-to-sql` subagent.

In [6]:
from text2sql_agent.agent import COORDINATOR_PROMPT, SQL_SUBAGENT_PROMPT

display(Markdown("### Coordinator system prompt"))
print(COORDINATOR_PROMPT)
display(Markdown("### Text-to-SQL subagent system prompt"))
print(SQL_SUBAGENT_PROMPT)

### Coordinator system prompt

You coordinate a conversational data analyst for the Chinook SQLite database.
Delegate every request that needs database facts or SQL to the `text-to-sql`
subagent using the task tool. You may use get_saved_result for follow-ups about
an existing result. Do not invent database facts or execute SQL yourself.

Return a FinalAnswer with a direct answer, the exact executed SQL and result ID
when present, material assumptions, and a concise interpretation. Do not expose
private chain of thought, tool payloads, or more than ten database rows.



### Text-to-SQL subagent system prompt

You are the isolated Chinook text-to-SQL analyst.

Before writing SQL, read `/project/semantic/chinook.osi.yaml` with a read limit
of at least 1000 lines, then load the relevant query-writing and
schema-exploration skills. The OSI model is authoritative. Use table/schema
tools only when it leaves a concrete ambiguity or appears inconsistent with the
live database. Use write_todos only for complex questions.

Semantic dataset and field names are conceptual identifiers, not SQL names.
Every SQL table must use the dataset's exact `source` value, and every SQL
column must use the field expression's exact physical value. For example, use
`InvoiceLine`, not `invoice_lines`, and `UnitPrice`, not `unit_price`.

Write exactly one read-only SQLite SELECT/CTE/set query and check it before
calling execute_sql. The execute_sql call pauses for human approval and may be
edited or rejected. On rejection, use the feedback to revise and submit a new
execute_sql call. Default ranked or list results to fi

In [7]:
print((PROJECT_ROOT / "AGENTS.md").read_text())

# Chinook Data Analyst

You are the coordinator for a conversational, human-reviewed data analyst.

## Operating model

- Delegate every database question to the `text-to-sql` subagent through `task`.
- Keep the user's conversational context, including references to prior result IDs.
- Use `get_saved_result` for follow-ups that can be answered from an existing result.
- Report only concise assumptions and interpretation—never private reasoning.
- Preserve the exact SQL and result ID returned by the SQL analyst.

## Analysis defaults

- The OSI model at `/project/semantic/chinook.osi.yaml` is the primary schema context.
- Simple ranked/list questions default to five rows unless the user requests another size.
- Complex questions should be planned with `write_todos`; simple questions should not.
- SQL must be approved by the human before execution. A rejection means revise the
  analysis and submit a new query for review.
- Full query results are application artifacts. Never copy more th

In [8]:
skill_paths = sorted((PROJECT_ROOT / "skills").glob("*/SKILL.md"))
for path in skill_paths:
    display(Markdown(f"### `{path.parent.name}` skill"))
    print(path.read_text())

### `query-writing` skill

---
name: query-writing
description: Write safe SQLite SELECT queries for Chinook analysis and submit them for human-reviewed execution.
---

# Query Writing

1. Read `/project/semantic/chinook.osi.yaml` first with a limit of at least
   1000 lines. Treat it as the primary source of tables, columns, joins, and
   metric definitions.
2. For a complex multi-step question, use `write_todos` before writing SQL.
   Skip todos for a straightforward lookup or aggregation.
3. Use exact physical names from OSI expressions. Query only needed columns.
4. Use `sql_db_query_checker` before `execute_sql`. Use schema tools only if
   OSI lacks necessary detail.
5. Call only `execute_sql` to run SQL. It pauses for human approval and may
   return an edited query result after review.

## SQL rules

- Exactly one read-only SQLite query: SELECT, WITH/CTE, or a set operation.
- No DML, DDL, transactions, PRAGMA, ATTACH, or multiple statements.
- Default list/ranking output to `LIMIT 5` unless the user spe

### `schema-exploration` skill

---
name: schema-exploration
description: Ground Chinook questions in its OSI semantic model, with live SQLite schema inspection only as fallback.
---

# Schema Exploration

1. Always read `/project/semantic/chinook.osi.yaml` before using database
   introspection. Use a read limit of at least 1000 lines.
2. Locate relevant datasets, physical sources, fields, relationships, metrics,
   AI instructions, and synonyms in that file.
3. Use `sql_db_list_tables` or `sql_db_schema` only to resolve a concrete gap
   or verify drift between the semantic model and the database.
4. Never probe the database with exploratory SQL when the semantic model
   already answers the question.

When explaining schema, distinguish physical names such as `InvoiceLine` from
logical OSI names such as `invoice_lines`. Use only declared relationship paths.
Playlist membership is not a sale.



### What “progressive disclosure” means here

A skill path is registered with the subagent, but the subagent reads the detailed `SKILL.md` only as it works. That keeps the initial prompt smaller while preserving specialized guidance. The SQL prompt also requests a read limit of at least 1,000 lines because Deep Agents' file-reading tool otherwise reads a small default slice.

The project filesystem is mounted virtually at `/project/`. Permissions are allowlisted for `AGENTS.md`, `semantic/**`, and `skills/**`, followed by catch-all deny rules. Permission order matters because the first matching rule wins.

## 5. Structured outputs are contracts

Free-form text is pleasant for a person but unreliable for application code. The specialist and coordinator use strict Pydantic contracts through LangChain's provider-native `ProviderStrategy`. Unknown fields are rejected.

- `QueryResult` is the small executor response visible to the model.
- `SQLAnalysisResult` is the specialist's interpretation.
- `FinalAnswer` is the coordinator's response to the UI.

In [12]:
from text2sql_agent.schemas import FinalAnswer, QueryResult, SQLAnalysisResult

for schema in (QueryResult, SQLAnalysisResult, FinalAnswer):
    display(Markdown(f"### `{schema.__name__}`"))
    properties = schema.model_json_schema()["properties"]
    display(pd.DataFrame([
        {
            "field": name,
            "type": spec.get("type", spec.get("anyOf", "nested")),
            "description": spec.get("description", ""),
        }
        for name, spec in properties.items()
    ]))

### `QueryResult`

,field,type,description
0,result_id,string,
1,executed_sql,string,
2,columns,array,
3,sample_rows,array,
4,row_count,integer,
5,truncated,boolean,
6,elapsed_ms,number,


### `SQLAnalysisResult`

,field,type,description
0,answer,string,
1,sql,"[{'type': 'string'}, {'type': 'null'}]",
2,result_id,"[{'type': 'string'}, {'type': 'null'}]",
3,row_count,"[{'maximum': 500, 'minimum': 0, 'type': 'integ...",
4,assumptions,array,
5,interpretation,string,


### `FinalAnswer`

,field,type,description
0,answer,string,
1,sql,"[{'type': 'string'}, {'type': 'null'}]",
2,result_id,"[{'type': 'string'}, {'type': 'null'}]",
3,assumptions,array,
4,interpretation,string,


## 6. SQL safety is defense in depth

Human review is valuable, but it is not the only safety control:

1. SQLGlot parses exactly one SQLite query expression.
2. DDL, DML, transactions, PRAGMA, ATTACH, and commands are rejected.
3. SQLite opens the file with `mode=ro`.
4. A SQLite authorizer denies mutation operations.
5. A progress handler enforces a time limit.
6. The executor fetches only 501 rows and stores at most 500.
7. The model sees at most 10 sample rows.

The exact reviewed SQL is executed. The executor never silently adds or changes a `LIMIT`.

In [13]:
from text2sql_agent.sql_tools import SQLValidationError, validate_readonly_sql

print(inspect.getsource(validate_readonly_sql))

def validate_readonly_sql(query: str) -> exp.Query:
    """Parse and validate one SELECT/CTE/set-operation query."""

    if not query or not query.strip():
        raise SQLValidationError("SQL cannot be empty.")
    try:
        statements = parse(query, read="sqlite")
    except ParseError as exc:
        raise SQLValidationError(f"Invalid SQLite SQL: {exc}") from exc
    statements = [statement for statement in statements if statement is not None]
    if len(statements) != 1:
        raise SQLValidationError("Exactly one SQL statement is required.")

    statement = statements[0]
    if not isinstance(statement, exp.Query):
        raise SQLValidationError(
            "Only read-only SELECT, CTE, and set-operation queries are allowed."
        )

    forbidden_names = {
        "Alter",
        "Analyze",
        "Attach",
        "Command",
        "Commit",
        "Copy",
        "Create",
        "Delete",
        "Detach",
        "Drop",
        "Execute",
        "Grant",
 

In [14]:
examples = {
    "simple SELECT": "SELECT Name FROM Artist LIMIT 5",
    "CTE": "WITH totals AS (SELECT SUM(Total) AS revenue FROM Invoice) SELECT * FROM totals",
    "set operation": "SELECT Name FROM Artist UNION SELECT Name FROM Genre",
    "write": "UPDATE Artist SET Name = 'unsafe' WHERE ArtistId = 1",
    "multiple statements": "SELECT 1; SELECT 2",
    "PRAGMA": "PRAGMA table_info(Artist)",
}

rows = []
for label, sql in examples.items():
    try:
        expression = validate_readonly_sql(sql)
        rows.append({"example": label, "accepted": True, "result": type(expression).__name__})
    except SQLValidationError as exc:
        rows.append({"example": label, "accepted": False, "result": str(exc)})
display(pd.DataFrame(rows))

,example,accepted,result
0,simple SELECT,True,Select
1,CTE,True,Select
2,set operation,True,Union
3,write,False,"Only read-only SELECT, CTE, and set-operation ..."
4,multiple statements,False,Exactly one SQL statement is required.
5,PRAGMA,False,"Only read-only SELECT, CTE, and set-operation ..."


## 7. The deterministic executor and result artifact

The next cell calls the guarded executor directly so you can study its data contract without paying for a model call. In the real agent, this same operation is wrapped as the `execute_sql` tool and is protected by HITL middleware.

Direct invocation here bypasses only the approval pause—not the parser, read-only connection, authorizer, timeout, or row cap.

In [15]:
from text2sql_agent.sql_tools import execute_query
from text2sql_agent.stores import ResultStore

lab_store = ResultStore()
lab_thread_id = "notebook-safety-lab"
lab_sql = """
SELECT g.Name AS genre, COUNT(t.TrackId) AS track_count
FROM Genre AS g
LEFT JOIN Track AS t ON t.GenreId = g.GenreId
GROUP BY g.GenreId, g.Name
ORDER BY track_count DESC, genre ASC
""".strip()

small_result = execute_query(
    database_path=settings.database_path,
    query=lab_sql,
    thread_id=lab_thread_id,
    result_store=lab_store,
    timeout_seconds=settings.sql_timeout_seconds,
)
display(pd.DataFrame(small_result.sample_rows))
display(small_result)

,genre,track_count
0,Rock,1297
1,Latin,579
2,Metal,374
3,Alternative & Punk,332
4,Jazz,130
5,TV Shows,93
6,Blues,81
7,Classical,74
8,Drama,64
9,R&B/Soul,61


QueryResult(result_id='e320e3b4-2506-4717-917a-418228cd8cfa', executed_sql='SELECT g.Name AS genre, COUNT(t.TrackId) AS track_count\nFROM Genre AS g\nLEFT JOIN Track AS t ON t.GenreId = g.GenreId\nGROUP BY g.GenreId, g.Name\nORDER BY track_count DESC, genre ASC', columns=['genre', 'track_count'], sample_rows=[{'genre': 'Rock', 'track_count': 1297}, {'genre': 'Latin', 'track_count': 579}, {'genre': 'Metal', 'track_count': 374}, {'genre': 'Alternative & Punk', 'track_count': 332}, {'genre': 'Jazz', 'track_count': 130}, {'genre': 'TV Shows', 'track_count': 93}, {'genre': 'Blues', 'track_count': 81}, {'genre': 'Classical', 'track_count': 74}, {'genre': 'Drama', 'track_count': 64}, {'genre': 'R&B/Soul', 'track_count': 61}], row_count=25, truncated=False, elapsed_ms=3.568083979189396)

In [16]:
artifact = lab_store.get(small_result.result_id, lab_thread_id)
page = lab_store.page(small_result.result_id, lab_thread_id, offset=10, limit=10)

print(f"Rows visible to the model: {len(small_result.sample_rows)}")
print(f"Rows saved for the application: {artifact.row_count}")
print(f"Second page rows: {len(page.rows)}")
display(pd.DataFrame(page.rows))

# Thread scoping prevents another conversation from reading the artifact.
try:
    lab_store.get(small_result.result_id, "another-thread")
except KeyError:
    print("Thread isolation check passed.")

Rows visible to the model: 10
Rows saved for the application: 25
Second page rows: 10


,genre,track_count
0,Reggae,58
1,Pop,48
2,Soundtrack,43
3,Alternative,40
4,Hip Hop/Rap,35
5,Electronica/Dance,30
6,Heavy Metal,28
7,World,28
8,Sci Fi & Fantasy,26
9,Easy Listening,24


Thread isolation check passed.


## 8. Build the real Deep Agent

This creates the same compiled graph used by FastAPI. It does **not** send a request to OpenAI.

Important constructor choices:

- the default general-purpose subagent is disabled;
- exactly one custom `text-to-sql` subagent is registered;
- the custom subagent gets its own tools, skills, permissions, prompt, and output schema;
- `execute_sql` has `approve`, `edit`, and `reject` interrupt decisions;
- `InMemorySaver` checkpoints paused state by `thread_id`; and
- the coordinator exposes only `get_saved_result` directly.

In [17]:
from langgraph.checkpoint.memory import InMemorySaver
from text2sql_agent.agent import build_agent

agent_result_store = ResultStore()
checkpointer = InMemorySaver()
agent = build_agent(
    settings,
    agent_result_store,
    checkpointer=checkpointer,
)

print("Agent name:", agent.name)
print("Compiled graph nodes:", sorted(agent.nodes))

Agent name: chinook-data-analyst
Compiled graph nodes: ['MemoryMiddleware.before_agent', 'PatchToolCallsMiddleware.before_agent', 'TodoListMiddleware.after_model', '__start__', 'model', 'tools']


In [ ]:
# This is the actual application constructor, shown as executable documentation.
print(inspect.getsource(build_agent))

## 9. Run the full agent and pause before SQL

Set `RUN_LIVE_AGENT = True` when you are ready. The first model run should stop at an `execute_sql` interrupt. It must not execute the database query yet.

The checkpoint key is the LangGraph `thread_id`. A resume must use the **same** configuration. `AgentContext` separately carries the application conversation/run identity into tools.

In [18]:
RUN_LIVE_AGENT = True  # Change to True to call OpenAI.

question = "Which five artists generated the most line-item revenue?"
live_thread_id = f"notebook-{uuid.uuid4()}"
live_run_id = f"run-{uuid.uuid4()}"
live_config = {"configurable": {"thread_id": live_thread_id}}

print("Question:", question)
print("Checkpoint thread:", live_thread_id)
print("Live call enabled:", RUN_LIVE_AGENT)

Question: Which five artists generated the most line-item revenue?
Checkpoint thread: notebook-220bf6b9-6628-458b-b206-981ebb3ac372
Live call enabled: True


In [19]:
from text2sql_agent.run_manager import _extract_approval
from text2sql_agent.sql_tools import AgentContext

first_output = None
approval = None

if RUN_LIVE_AGENT:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Add OPENAI_API_KEY to .env before enabling the live lab.")
    live_context = AgentContext(thread_id=live_thread_id, run_id=live_run_id)
    first_output = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=live_config,
        context=live_context,
        version="v2",
    )
    interrupts = list(getattr(first_output, "interrupts", []))
    approval = _extract_approval(interrupts)
    print("The graph paused before execution.")
    print("Allowed decisions:", approval.allowed_decisions)
    print("\nGenerated SQL:\n")
    print(approval.query)
else:
    print("Skipped. Set RUN_LIVE_AGENT=True, then rerun this cell.")

The graph paused before execution.
Allowed decisions: ['approve', 'edit', 'reject']

Generated SQL:

WITH artist_revenue AS (
  SELECT
    ar.Name AS artist_name,
    SUM(il.UnitPrice * il.Quantity) AS line_item_revenue
  FROM InvoiceLine il
  JOIN Track t ON il.TrackId = t.TrackId
  JOIN Album al ON t.AlbumId = al.AlbumId
  JOIN Artist ar ON al.ArtistId = ar.ArtistId
  GROUP BY ar.ArtistId, ar.Name
)
SELECT
  artist_name,
  line_item_revenue
FROM artist_revenue
ORDER BY line_item_revenue DESC, artist_name ASC
LIMIT 5;


In [23]:
for m in first_output.value["messages"]:
    m.pretty_print()

================================ Human Message =================================

Which five artists generated the most line-item revenue?
================================== Ai Message ==================================
Name: chinook-data-analyst
Tool Calls:
  task (call_R2LoJXRXp9DkVY3EB1Jq7jSX)
 Call ID: call_R2LoJXRXp9DkVY3EB1Jq7jSX
  Args:
    description: Use the Chinook SQLite database via the text-to-sql workflow to answer: 'Which five artists generated the most line-item revenue?' Need concise business answer with exact executed SQL and result_id. Follow the OSI/schema context, determine the correct revenue definition based on line items (likely invoice line quantity*unitprice rolled up to artists through track/album). Ask human review/approval if required by your workflow. Return only: final answer summary, exact SQL executed, result id, assumptions, and a brief interpretation. Limit to top five rows in the returned data.
    subagent_type: text-to-sql


### Review it like an analyst

Before approving, ask:

- Does the SQL answer the actual question?
- Are joins consistent with the OSI relationships?
- Is revenue defined at the correct grain?
- Could a join duplicate invoice totals?
- Is the result ordered and limited as requested?

The decision translator validates both approved and edited SQL, then creates LangGraph's required shape: `Command(resume={"decisions": [...]})`. For an edit, the new call is supplied under `edited_action`; the reviewed action order is preserved.

In [ ]:
from text2sql_agent.run_manager import decisions_to_command
from text2sql_agent.schemas import Decision

# Choose one: "approve", "edit", or "reject".
REVIEW_ACTION = "approve"
EDITED_SQL = approval.query if approval else "SELECT Name FROM Artist LIMIT 5"
REJECTION_FEEDBACK = "Revise the query to use line-level revenue and return five artists."

resume_command = None
if approval:
    if REVIEW_ACTION == "approve":
        decision = Decision(action="approve")
    elif REVIEW_ACTION == "edit":
        decision = Decision(action="edit", edited_sql=EDITED_SQL)
    elif REVIEW_ACTION == "reject":
        decision = Decision(action="reject", feedback=REJECTION_FEEDBACK)
    else:
        raise ValueError("REVIEW_ACTION must be approve, edit, or reject")
    resume_command = decisions_to_command(approval, [decision])
    print(resume_command)
else:
    print("No pending approval. Run the live cell first.")

In [ ]:
completed_output = None
final_answer = None

if RUN_LIVE_AGENT and resume_command is not None:
    live_context = AgentContext(thread_id=live_thread_id, run_id=live_run_id)
    completed_output = agent.invoke(
        resume_command,
        config=live_config,
        context=live_context,
        version="v2",
    )
    new_interrupts = list(getattr(completed_output, "interrupts", []))
    if new_interrupts:
        next_approval = _extract_approval(new_interrupts)
        print("The agent requested another review cycle.")
        print(next_approval.query)
    else:
        state = completed_output.value if hasattr(completed_output, "value") else completed_output
        final_answer = state.get("structured_response")
        display(final_answer)
else:
    print("Skipped until the live lab is enabled and an approval exists.")

In [ ]:
if final_answer and final_answer.result_id:
    saved = agent_result_store.get(final_answer.result_id, live_thread_id)
    print("Exact executed SQL:\n")
    print(saved.executed_sql)
    print(f"\nStored rows: {saved.row_count}; truncated: {saved.truncated}")
    display(pd.DataFrame(saved.rows, columns=saved.columns))
else:
    print("A completed live SQL run will display its full stored artifact here.")

### What happens on rejection?

A rejection does not execute SQL. Feedback is returned to the subagent, which can revise its plan and submit another `execute_sql` action. That produces a **new interrupt**. Repeat the review cells with the new approval. This is why the UI supports repeated approval cycles rather than assuming one review per message.

### What happens on edit?

The edited SQL is validated before resume. If valid, the middleware replaces the pending tool arguments. The exact edited text is the SQL executed and later shown in the UI.

## 10. Observability without exposing hidden reasoning

The production run manager consumes `astream_events(version="v3")` internally. It converts raw tool lifecycle events into safe labels such as “Inspecting the OSI semantic model” and “Checking generated SQL.” It deliberately does **not** send prompts, hidden reasoning, tool payloads, or raw database rows to the browser.

In [25]:
from text2sql_agent.run_manager import _activity_for_tool

sample_tool_events = [
    ("task", {}),
    ("read_file", {"file_path": "/project/semantic/chinook.osi.yaml"}),
    ("read_file", {"file_path": "/project/skills/query-writing/SKILL.md"}),
    ("write_todos", {}),
    ("sql_db_schema", {}),
    ("sql_db_query_checker", {}),
    ("execute_sql", {"query": "SELECT secret-looking-payload"}),
]
display(pd.DataFrame([
    {"tool": name, "sanitized_activity": _activity_for_tool(name, payload)}
    for name, payload in sample_tool_events
]))

,tool,sanitized_activity
0,task,"(subagent, Delegating to the text-to-SQL analyst)"
1,read_file,"(semantic, Inspecting the OSI semantic model)"
2,read_file,"(skill, Loading an analysis skill)"
3,write_todos,"(planning, Planning the analysis)"
4,sql_db_schema,"(schema, Checking live table schema as fallback)"
5,sql_db_query_checker,"(sql_check, Checking generated SQL)"
6,execute_sql,"(execution, Executing approved SQL)"


## 11. How the notebook maps to FastAPI and Streamlit

The browser does not call the graph directly. FastAPI owns process-local conversations, runs, checkpoints, and results. Streamlit starts a run and polls it without streaming model tokens.

```text
queued → running → approval_required → running → completed
                    │                     │
                    └── reject/replan ────┘
Any active state can become failed.
```

At `approval_required`, Streamlit renders the pending SQL in an editable text area. The user's decision is posted to FastAPI, translated into a LangGraph `Command`, and resumed against the same checkpoint thread.

In [ ]:
from text2sql_agent.api import Services, create_app

tutorial_app = create_app(Services(settings=settings))
route_rows = []
for route in tutorial_app.routes:
    if getattr(route, "path", "").startswith(("/api", "/health")):
        route_rows.append({
            "methods": ", ".join(sorted(route.methods or [])),
            "path": route.path,
            "name": route.name,
        })
display(pd.DataFrame(route_rows))

## 12. Core design lessons

1. **Context engineering beats blind exploration.** Put stable business meaning in an OSI semantic model and reserve live schema tools for fallback.
2. **Isolate specialists.** Give each custom subagent only the prompt, skills, tools, permissions, and response contract it needs.
3. **Use progressive disclosure.** Keep broad prompts concise; load procedural skills when relevant.
4. **Checkpoint before human decisions.** HITL only works reliably when pause and resume share the same checkpointer and thread ID.
5. **Treat structured output as an interface.** Models produce typed data that application code can validate and render.
6. **Keep data artifacts outside context.** Give the model a result ID and a small sample; give the UI the complete capped artifact.
7. **Use deterministic safety controls.** Prompts and human review complement parsing, read-only mode, authorization, timeouts, and caps; they do not replace them.
8. **Sanitize observability.** Show useful progress without leaking prompts, hidden reasoning, SQL payloads, or result rows.
9. **Make assumptions explicit.** The answer should state material assumptions and a concise interpretation alongside the data.
10. **Test lifecycle edges.** Approval, edits, rejection loops, same-thread resume, truncation, thread isolation, refresh, and concurrent-run rejection matter as much as the happy path.

## 13. Suggested exercises

Try these in order:

1. Change the direct executor query and observe the 10-row model sample versus the full artifact.
2. Add a harmless semicolon and then a second statement to see the validator's boundary.
3. Enable the live lab and approve the generated SQL unchanged.
4. Start a fresh live thread, edit the SQL before execution, and confirm `executed_sql` exactly matches your edit.
5. Start another fresh thread, reject with specific feedback, and inspect the next proposed query.
6. Ask a follow-up about a saved `result_id` and observe `get_saved_result` instead of a fresh query where appropriate.
7. Modify a synonym in the OSI file and test how the agent interprets that phrase.
8. Write a complex analytical question and watch for `write_todos` in the Streamlit activity panel.

For production, the main missing pieces are durable storage/checkpoints, authentication and tenant isolation, stronger resource governance, deployment hardening, evaluation datasets, and operational tracing.

## 14. Verify the implementation

The final cell runs the automated suite. The live OpenAI smoke test remains skipped unless its explicit test conditions are enabled.

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()

## 15. Continue learning

- [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents subagents](https://docs.langchain.com/oss/python/deepagents/subagents)
- [Deep Agents event streaming](https://docs.langchain.com/oss/python/deepagents/event-streaming)
- [LangChain human-in-the-loop middleware](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangChain structured output](https://docs.langchain.com/oss/python/langchain/structured-output)
- [Datawhale Deep Agents in Action: task planning](https://datawhalechina.github.io/deepagents-in-action/chapters/ch04-task-planning/)
- [Datawhale Deep Agents in Action: skills](https://datawhalechina.github.io/deepagents-in-action/chapters/ch07-skills/)
- [Datawhale Deep Agents in Action: human in the loop](https://datawhalechina.github.io/deepagents-in-action/chapters/ch09-human-in-the-loop/)
- [Apache Ossie/OSI semantic-model specification](https://github.com/apache/ossie/blob/main/core-spec/spec.md)
- [OpenAI GPT-5.4 mini](https://developers.openai.com/api/docs/models/gpt-5.4-mini)

The local `.codex/skills/langchain-dev-guide` also records version-sensitive Deep Agents and LangGraph engineering pitfalls used while building this tutorial.

In [3]:
import sqlite3
import pandas as pd

# Connect directly to the downloaded file (no server setup needed)
conn = sqlite3.connect('/Users/charlie/Downloads/dev_20240627/dev_databases/financial/financial.sqlite')

# Run a query and load it directly into a Pandas DataFrame
df = pd.read_sql("SELECT * FROM trans LIMIT 5", conn)

print(df)
conn.close()

   trans_id  account_id        date    type      operation  amount  balance  \
0         1           1  1995-03-24  PRIJEM          VKLAD    1000     1000   
1         5           1  1995-04-13  PRIJEM  PREVOD Z UCTU    3679     4679   
2         6           1  1995-05-13  PRIJEM  PREVOD Z UCTU    3679    20977   
3         7           1  1995-06-13  PRIJEM  PREVOD Z UCTU    3679    26835   
4         8           1  1995-07-13  PRIJEM  PREVOD Z UCTU    3679    30415   

  k_symbol bank     account  
0     None  NaN         NaN  
1     None   AB  41403269.0  
2     None   AB  41403269.0  
3     None   AB  41403269.0  
4     None   AB  41403269.0  
